# Day 2 — NumPy: Vectorization & Broadcasting with NZ City Distances

## Objective
Compare pure Python and NumPy implementations of the Haversine great-circle 
distance formula, and practice core NumPy operations: broadcasting, boolean 
indexing, and `np.where`.

## Prerequisites
- **Haversine Formula**: Computes distance between two points on a sphere.
  - hav(θ) = sin(θ/2)^2 = 1 - cos(θ) = hav(lat2-lat1) + cos(lat1) * cos(lat2) * hav(lon2-lon1)
  - θ = distance in radians between two points = 2 * atan2(sqrt(1-hav(θ)), sqrt(hav(θ)))
  - R = Earth radius radius (6371 km)
  - D = distance between two points in km = R * θ
- **Broadcasting**: NumPy's ability to apply operations to arrays of different shapes.
- **Boolean Indexing**: Filter arrays based on conditions.
- **`np.where`**: Conditionally assign values to arrays.

## Dataset
5 major NZ cities: Auckland, Wellington, Christchurch, Hamilton, Tauranga.
Coordinates in decimal degrees (WGS84).

## Tasks
- **Task 1**: Pure Python Haversine — nested loops, 5×5 distance matrix
- **Task 2**: NumPy vectorized Haversine — broadcasting, no explicit loops
- **Task 3**: Timing comparison (pure Python vs NumPy, scaled to 1000 cities)
- **Task 4**: Boolean indexing — filter city pairs with distance 300–500 km
- **Task 5**: `np.where` — label distances as local / regional / interisland


In [ ]:
# Public Constants
R = 6371.0  # Earth radius in km

# NZ 5 major cities: name, lat (deg), lon (deg)
cities = [
    ("Auckland", -36.85, 174.76),
    ("Wellington", -41.29, 174.78),
    ("Christchurch", -43.53, 172.63),
    ("Hamilton", -37.78, 175.28),
    ("Tauranga", -37.69, 176.17),
]

def haversine(lat1, lon1, lat2, lon2):
    """Pure Python Haversine distance in km."""
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    hav = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )
    theta = 2 * math.atan2(math.sqrt(hav), math.sqrt(1 - hav))
    return R * theta

In [ ]:
# Task 1: Pure Python Haversine — nested loops, 5×5 distance matrix
import math


def task1_result():
    n = len(cities)
    dist_matrix = [[0.0 for _ in range(n)] for _ in range(n)]

    for i in range(n):
        for j in range(n):
            _, lat1, lon1 = cities[i]
            _, lat2, lon2 = cities[j]
            dist_matrix[i][j] = haversine(lat1, lon1, lat2, lon2)

    return dist_matrix


def task1_print():
    dist_matrix = task1_result()
    # Print Result
    row = ""
    for i, (nameF, _, _) in enumerate(cities):
        for j, (nameT, _, _) in enumerate(cities):
            row += f"{nameF} to {nameT}: {dist_matrix[i][j]:.2f} km \n"
    print(row)


task1_print()

In [ ]:
# Task 2: NumPy vectorized Haversine — broadcasting, no explicit loops
import numpy as np


def task2_result():
    cities_np = np.array([[lat, lon] for _, lat, lon in cities])

    lats = np.radians(cities_np[:, 0])
    lons = np.radians(cities_np[:, 1])

    print("vectorized", lats)
    print("broadcasted (5, 1) \n", lats[:, None])
    print("broadcasted (1, 5) \n", lats[None, :])

    dlat = lats[:, np.newaxis] - lats[np.newaxis, :]
    dlon = lons[:, np.newaxis] - lons[np.newaxis, :]

    print("broadcasted (5, 5) \n", dlat)

    hav = (
        np.sin(dlat / 2) ** 2
        + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon / 2) ** 2
    )
    theta = 2 * np.arctan2(np.sqrt(hav), np.sqrt(1 - hav))

    dist_matrix_np = R * theta
    return dist_matrix_np


np.set_printoptions(precision=2)
print(task2_result())

In [ ]:
# Task 3: Timing comparison (pure Python vs NumPy, scaled to 1000 cities)
import time


cities_big = cities * 200


def task3_py_result():
    n_big = len(cities_big)
    dist_matrix_big = [[0.0 for _ in range(n_big)] for _ in range(n_big)]
    for i in range(n_big):
        for j in range(n_big):
            _, lat1, lon1 = cities_big[i]
            _, lat2, lon2 = cities_big[j]
            dist_matrix_big[i][j] = R * haversine(lat1, lon1, lat2, lon2)
    return dist_matrix_big


def task3_np_result():
    cities_np = np.array([[lat, lon] for _, lat, lon in cities_big])
    lats = np.radians(cities_np[:, 0])
    lons = np.radians(cities_np[:, 1])

    dlat = lats[:, None] - lats[None, :]
    dlon = lons[:, None] - lons[None, :]

    hav = (
        np.sin(dlat / 2) ** 2
        + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon / 2) ** 2
    )
    theta = 2 * np.arctan2(np.sqrt(hav), np.sqrt(1 - hav))
    dist_matrix_np = R * theta

    return dist_matrix_np


def task3_result():
    py_time = time.perf_counter()
    task3_py_result()
    py_time = time.perf_counter() - py_time
    np_time = time.perf_counter()
    task3_np_result()
    np_time = time.perf_counter() - np_time
    # Print the results
    print(f"Pure Python time: {py_time:.4f} seconds")
    print(f"NumPy time: {np_time:.4f} seconds")
    print(f"Speedup: {py_time / np_time:.2f}")


task3_result()

In [ ]:
# Task 4: Boolean indexing — filter city pairs with distance 300–500 km
dist_matrix_np = task2_result()
mask = (dist_matrix_np >= 300) & (dist_matrix_np <= 500)
pairs = np.where(mask)
print(pairs)

for i, j in list(zip(*pairs)):
    if i < j:  # avoid duplicates (symmetric matrix)
        print(f"{cities[i][0]} to {cities[j][0]}: {dist_matrix_np[i][j]:.2f} km")

In [ ]:
# Task 5: `np.where` — label distances as local / regional / interisland
n = len(cities)
labels = np.where(
    dist_matrix_np == 0,
    "self",
    np.where(
        dist_matrix_np < 100,
        "local",
        np.where(dist_matrix_np <= 400, "regional", "interisland"),
    ),
)

# Print as labeled table
header = "   ".join(f"{cities[i][0]:>12s}" for i in range(n))
print(f"{'':12s} {header}")

for i in range(n):
    row = "   ".join(f"{labels[i][j]:>12s}" for j in range(n))
    row = f"{cities[i][0]:>12s} {row}"
    print(row)